In [27]:
import re
import pandas as pd
from clean_recipes import load_recipes, handle_missing, clean_text, enforce_id_integrity

df = load_recipes("recipes.csv")
df = handle_missing(df)
df = clean_text(df)
df = enforce_id_integrity(df)

Initial shape: (522517, 4)

<class 'pandas.DataFrame'>
RangeIndex: 522517 entries, 0 to 522516
Data columns (total 4 columns):
 #   Column                 Non-Null Count   Dtype
---  ------                 --------------   -----
 0   RecipeId               522517 non-null  int64
 1   Name                   522517 non-null  str  
 2   RecipeCategory         521766 non-null  str  
 3   RecipeIngredientParts  522517 non-null  str  
dtypes: int64(1), str(3)
memory usage: 15.9 MB

No missing RecipeId values.
No duplicate RecipeId values.


In [28]:
df = df[df["Name"].str.len() >= 3]
df = df.drop_duplicates(subset=["Name", "RecipeIngredientParts"], keep="first")
df["Name"] = df["Name"].str.replace("&amp;", "&", regex=False)
df = df.reset_index(drop=True)
print(df.shape)

(520377, 4)


In [29]:
kw = pd.read_csv("recipes.csv", usecols=["RecipeId", "Keywords"])
df = df.merge(kw, on="RecipeId", how="left")
df["Keywords"] = df["Keywords"].fillna("")

def parse_r_list(s):
    return re.findall(r'"([^"]*)"', s)

df["kw_list"] = df["Keywords"].apply(parse_r_list)
df["ing_list"] = df["RecipeIngredientParts"].apply(parse_r_list)
df["ingredients_text"] = df["ing_list"].apply(", ".join)
df[["Name", "kw_list", "ing_list"]].head()

,Name,kw_list,ing_list
0,Low-Fat Berry Blue Frozen Dessert,"[Dessert, Low Protein, Low Cholesterol, Health...","[blueberries, granulated sugar, vanilla yogurt..."
1,Biryani,"[Chicken Thigh & Leg, Chicken, Poultry, Meat, ...","[saffron, milk, hot green chili peppers, onion..."
2,Best Lemonade,"[Low Protein, Low Cholesterol, Healthy, Summer...","[sugar, lemons, rind of, lemon, zest of, fresh..."
3,Carina's Tofu-Vegetable Kebabs,"[Beans, Vegetable, Low Cholesterol, Weeknight,...","[extra firm tofu, eggplant, zucchini, mushroom..."
4,Cabbage Soup,"[Low Protein, Vegan, Low Cholesterol, Healthy,...","[plain tomato juice, cabbage, onion, carrots, ..."


In [30]:
print(df["ing_list"][2])

['sugar', 'lemons, rind of', 'lemon, zest of', 'fresh water', 'fresh lemon juice']


In [31]:
(df["ing_list"].str.len() == 0).sum()

np.int64(2189)

In [32]:
clear = {
 "dessert": ["Dessert","Pie","Bar Cookie","Drop Cookies","Candy","Frozen Desserts","Cheesecake",
             "Tarts","Gelatin","Ice Cream","Chocolate Chip Cookies","Lemon Cake","Peanut Butter Pie",
             "Bread Pudding","Apple Pie","Key Lime Pie","Coconut Cream Pie","Desserts Fruit"],
 "beverage": ["Beverages","Punch Beverage","Smoothies","Shakes","Margarita"],
 "soup": ["Chowders","Clear Soup","Stew","Gumbo","Bean Soup","Ham And Bean Soup","Black Bean Soup",
          "Potato Soup","Broccoli Soup","Mushroom Soup","Soups Crock Pot","Stocks"],
 "sauce/condiment": ["Sauces","Salad Dressings","Spreads","Chutneys","Jellies","Spaghetti Sauce",
                     "Turkey Gravy","Summer Dip"],
 "bread": ["Breads","Quick Breads","Yeast Breads","Scones","Sourdough Breads","Buttermilk Biscuits","Wheat Bread"],
 "breakfast": ["Breakfast","Brunch","Oatmeal","Breakfast Eggs","Breakfast Casseroles"],
 "appetizer/snack": ["Gluten Free Appetizers"],
 "main course": ["Savory Pies","Curries","Meatloaf","Meatballs","Pot Pie","Pot Roast","Stir Fry",
                 "Main Dish Casseroles","Macaroni And Cheese","Spaghetti","Penne","Pasta Shells",
                 "Manicotti","Steak","Roast Beef"],
 "non-food": ["Bath/Beauty","Household Cleaner","Homeopathy/Remedies"],
}

likely = {
 "main course": ["One Dish Meal","Chicken","Meat","Pork","Chicken Breast","Poultry","Lamb/Sheep","Ham",
                 "Whole Chicken","Chicken Thigh & Leg","Tuna","Veal","Halibut","Tilapia","Turkey Breasts",
                 "Deer","Mussels","Whole Turkey","Lobster","Catfish","Crab","Trout","Wild Game",
                 "Beef Organ Meats","Crawfish","Squid","Mahi Mahi","Duck","Chicken Livers","Duck Breasts",
                 "Orange Roughy","Whole Duck","Rabbit","Whitefish","Quail","Pheasant","Goose","Octopus",
                 "Elk","Moose","Bear","Beef Liver","Roast","Bass","Perch","Oysters","Fish Tuna",
                 "Fish Salmon","Roast Beef Crock Pot","Chicken Crock Pot"],
 "side dish": ["Vegetable","Potato","Beans","Onions","Rice","Grains","Corn","Yam/Sweet Potato","Spinach",
               "Greens","Peppers","Cauliflower","Black Beans","Lentil","Long Grain Rice","White Rice",
               "Brown Rice","Short Grain Rice","Medium Grain Rice","Mashed Potatoes","Collard Greens",
               "Chard","Artichoke","Soy/Tofu","Tempeh"],
}

In [33]:
clear_map  = {cat: t for t, cats in clear.items()  for cat in cats}
likely_map = {cat: t for t, cats in likely.items() for cat in cats}

print(len(clear_map), len(likely_map))

74 75


In [34]:
all_cats = set(df["RecipeCategory"])
print([c for c in list(clear_map) + list(likely_map) if c not in all_cats])

[]


In [35]:
df["dish_type"] = df["RecipeCategory"].map(clear_map)
df["confidence"] = "high"
df["source"] = "category"

In [36]:
unresolved = df["dish_type"].isna()
df.loc[unresolved, "dish_type"] = df.loc[unresolved, "RecipeCategory"].map(likely_map)

likely_rows = unresolved & df["dish_type"].notna()
df.loc[likely_rows, "confidence"] = "medium"

In [37]:
still_empty = df["dish_type"].isna()
df.loc[still_empty, "confidence"] = "needs review"
df.loc[still_empty, "source"] = "none"

In [38]:
print(df["confidence"].value_counts())
print(df["dish_type"].value_counts(dropna=False))

confidence
high            224745
medium          171067
needs review    124565
Name: count, dtype: int64
dish_type
NaN                124565
main course        116940
dessert            100082
side dish           69672
bread               29985
sauce/condiment     22738
beverage            22681
breakfast           21818
soup                10866
non-food             1029
appetizer/snack         1
Name: count, dtype: int64


In [39]:
df[df["confidence"] == "needs review"]["RecipeCategory"].value_counts().head(15)

RecipeCategory
Lunch/Snacks       32512
< 60 Mins           9653
< 30 Mins           8964
Cheese              8437
< 15 Mins           6607
Low Protein         6601
< 4 Hours           4938
Low Cholesterol     3783
European            3676
Very Low Carbs      3355
Fruit               2560
Asian               2189
Mexican             2110
Healthy             1710
Weeknight           1334
Name: count, dtype: int64

In [40]:
rules = [
    # exceptions first: these contain "cake" or "roll" but are not desserts or breads
    ("main course",     r"\b(?:crab|fish|salmon|tuna|potato|rice|shrimp|chicken|veggie|vegetable) cakes?\b|\b(?:egg|spring) rolls?\b"),
    ("soup",            r"\b(?:soups?|chowder|stew|bisque|broth)\b"),
    ("salad",           r"\bsalads?\b"),
    ("main course",     r"\b(?:pot pie|shepherd|casserole|lasagna|chili|curry|stir fry|burgers?|tacos?|pizza|sandwich(?:es)?)\b"),
    ("dessert",         r"\b(?:desserts?|cakes?|cookies?|brownies|pudding|cobbler|fudge|ice cream|cupcakes?|pies?|tarts?|mousse|custard)\b"),
    ("beverage",        r"\b(?:beverages?|smoothies?|lemonade|cocktail|punch|milkshake|tea|coffee|margarita)\b"),
    ("sauce/condiment", r"\b(?:sauces?|dressing|dips?|salsa|gravy|marinade|spread|relish|chutney)\b"),
    ("bread",           r"\b(?:bread|rolls?|biscuits?|muffins?|scones?|bagels?|focaccia)\b"),
    ("breakfast",       r"\b(?:pancakes?|waffles?|omelet|omelette|french toast|granola|breakfast)\b"),
    ("appetizer/snack", r"\b(?:appetizers?|snacks?|finger food)\b"),
]

def match_rules(text):
    text = text.str.lower()
    result = pd.Series("", index=text.index, dtype="object")
    for dish, pattern in rules:
        hit = (result == "") & text.str.contains(pattern, regex=True)
        result.loc[hit] = dish
    return result

In [41]:
test = pd.Series(["Cabbage Soup", "Crab Cakes", "Banana Bread", "Chocolate Lava Cake", "Greek Salad", "Mystery Dish"])
print(match_rules(test).tolist())

['soup', 'main course', 'bread', 'dessert', 'salad', '']


In [42]:
name_type = match_rules(df["Name"])
kw_type = match_rules(df["kw_list"].apply(" | ".join))
med = df["confidence"] == "medium"

print((med & (name_type != "")).sum(), "medium rows where the name gives a signal")
print((med & (kw_type != "")).sum(), "medium rows where the keywords give a signal")
print((med & (name_type == "") & (kw_type != "")).sum(), "medium rows where only the keywords give a signal")

62608 medium rows where the name gives a signal
4718 medium rows where the keywords give a signal
3162 medium rows where only the keywords give a signal


In [43]:
name_type = match_rules(df["Name"])

# a specific word in the name overrides a "likely" category (Cabbage Soup under Vegetable)
override = (df["confidence"] == "medium") & (name_type != "") & (name_type != "main course")
df.loc[override, "dish_type"] = name_type[override]
df.loc[override, "source"] = "name (override)"

# unresolved recipes: try the name first
by_name = df["dish_type"].isna() & (name_type != "")
df.loc[by_name, "dish_type"] = name_type[by_name]
df.loc[by_name, "confidence"] = "medium"
df.loc[by_name, "source"] = "name"

# then the Keywords tags
kw_type = match_rules(df["kw_list"].apply(" | ".join))
by_kw = df["dish_type"].isna() & (kw_type != "")
df.loc[by_kw, "dish_type"] = kw_type[by_kw]
df.loc[by_kw, "confidence"] = "low"
df.loc[by_kw, "source"] = "keywords"

# nothing matched
df["dish_type"] = df["dish_type"].fillna("unknown")

In [44]:
df["conflict"] = (df["source"] == "category") & (name_type != "") & (name_type != df["dish_type"])

In [45]:
print(df["confidence"].value_counts())
print(df["dish_type"].value_counts())
print(df["conflict"].sum(), "conflicts")

confidence
medium          228497
high            224745
needs review     66692
low                443
Name: count, dtype: int64
dish_type
main course        114706
dessert            105280
unknown             66692
sauce/condiment     47216
side dish           46823
bread               33659
soup                31776
salad               25629
beverage            23612
breakfast           22726
appetizer/snack      1229
non-food             1029
Name: count, dtype: int64
29390 conflicts


In [46]:
review = df[df["confidence"] == "needs review"]
print(review["RecipeCategory"].value_counts().head(20))
review[["Name", "RecipeCategory"]].sample(20, random_state=1)

RecipeCategory
Lunch/Snacks       17020
< 60 Mins           4967
< 30 Mins           4780
Cheese              4619
Low Protein         3159
< 15 Mins           3136
Very Low Carbs      2438
European            2298
< 4 Hours           2290
Low Cholesterol     1829
Fruit               1319
Asian               1310
Healthy             1154
Mexican              996
High Protein         913
Weeknight            696
Apple                669
Kid Friendly         536
Vegan                463
Pineapple            447
Name: count, dtype: int64


,Name,RecipeCategory
456773,Grilled Tilapia With Chimichurri Mayo,South American
441628,Deviled Almonds,Very Low Carbs
411083,Greek Seafood Feast for the Eyes and Palate,Very Low Carbs
499582,Chipotle Deviled Eggs and Bacon,< 60 Mins
412892,Whole Pears Roasted in Garlic,< 60 Mins
483075,Peanut Butter Energy Bites,< 15 Mins
4252,Barbecue Ribs,Weeknight
308464,Cinnamon Sug' Sweet Potato Sticks,Low Protein
483826,Kool-Aid Popcorn,< 60 Mins
235641,Basil Butter,Very Low Carbs


In [47]:
name_type = match_rules(df["Name"])
conf = df[df["conflict"]].copy()
conf["name_type"] = name_type[conf.index]

print(conf.groupby(["dish_type", "name_type"]).size().sort_values(ascending=False).head(12))
conf[["Name", "RecipeCategory", "dish_type", "name_type"]].sample(20, random_state=1)

dish_type        name_type      
side dish        main course        6550
sauce/condiment  salad              2585
bread            dessert            2433
breakfast        main course        1666
                 dessert            1377
main course      sauce/condiment    1368
breakfast        bread              1367
main course      dessert            1188
dessert          salad              1045
                 bread              1008
                 main course         924
                 sauce/condiment     829
dtype: int64


,Name,RecipeCategory,dish_type,name_type
250738,Ya Gotta Try It Breakfast Casserole,Breakfast,breakfast,main course
234730,Key Lime Pie Martini,Beverages,beverage,dessert
416505,&quot;olive Garden&quot; Salad and Dressing Re...,Salad Dressings,sauce/condiment,salad
99732,Old-Fashioned Hot Fudge Sauce,Sauces,sauce/condiment,dessert
457035,Strawberry Jello Salad (Healthy),Gelatin,dessert,salad
243391,Apple Top Coffee Cake,Breads,bread,dessert
279607,Fat Free Sugar Free Blueberry Muffins,Breakfast,breakfast,bread
499969,Curry Dip,Sauces,sauce/condiment,main course
191760,Grandpa Rondina's Meatballs and Sauce,Spaghetti,main course,sauce/condiment
162763,Shrimp Pasta Salad With Dill and Parsley,Pasta Shells,main course,salad


In [48]:
pairs = [("side dish","main course"), ("bread","dessert"), ("breakfast","main course"),
         ("breakfast","dessert"), ("main course","sauce/condiment"), ("breakfast","bread"),
         ("main course","dessert"), ("dessert","salad")]

for a, b in pairs:
    sub = conf[(conf["dish_type"] == a) & (conf["name_type"] == b)]
    print(f"\n{a} (category) vs {b} (name): {len(sub)}")
    print(sub[["Name", "RecipeCategory"]].sample(8, random_state=1).to_string(index=False))


side dish (category) vs main course (name): 6550
                                                   Name RecipeCategory
                            Green Bean & Corn Casserole          Beans
                                  Melanie's Pizza Dough         Grains
                                Cheese - Corn Casserole           Corn
                      Roasted Corn and Black Bean Chili    Black Beans
                                      Tim Hortons Chili         Onions
Falafel W/ Avocado Spread - Veggie Burgers W/ Guacamole          Beans
                                        Baked Egg Rolls      Vegetable
                                       Vegetarian Chili          Beans

bread (category) vs dessert (name): 2433
                                             Name RecipeCategory
Bad for Hips, Good for Lips Blueberry Coffee Cake         Breads
                     Cinnamon Coffee Cake Muffins   Quick Breads
               Delicious Graham Apple Coffee Cake         Breads
         